# ntv_double_selec

> 对应代码：`EconometricsCode/code_in_notes/Chap.10/ntv_double_selec.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.10`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.10")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们加载同一份 NTV 面板数据并声明面板结构，为使用滞后算子 L. 做准备。

In [ ]:
%%stata
clear
set more off
use ../datasets/NTV_Aggregate_Data_reshaped.dta
tsset tik_id year

为了以更灵活的方式控制社会经济状况，我们把滞后对数人口与滞后对数工资各取 1 至 5 次多项式；循环一次生成 10 个高阶项，避免逐个手写。

In [ ]:
%%stata
// 社会经济变量以及人口、工资的多项式
forvalues i=1/5{
    gen lag_log_pop_`i'=L.logpop^`i'
    gen lag_log_wage_`i'=L.log_wage^`i'
}

再补上护士数与人均医生数的一期滞后，连同两组多项式一起收入宏 E_controls，构成一个高维的候选控制变量集合。

In [ ]:
%%stata
gen lag_nurses=L.nurses
gen lag_doctors_pc=L.doctors_pc
local E_controls "lag_log_pop_* lag_log_wage_* lag_nurses lag_doctors_pc"

作为基准，我们先用 reghdfe 在 1999 年截面上做全控制回归：地区固定效应用 a(region) 吸收，标准误按地区聚类。这里没有做任何变量选择，所有多项式项一律进入回归。

In [ ]:
%%stata
reghdfe Votes_SPS_ Watch_probit `E_controls' if year==1999, a(region) cluster(region)

接着用 dsregress 做双重选择（double selection）Lasso：分别对结果方程与处理方程做 Lasso 选择，把两次被选中的控制变量取并集后再做 OLS，从而在遗漏重要控制的风险下仍对处理效应做有效推断；selec(cv) 表示用交叉验证选取惩罚参数，controls((i.region) ...) 把地区虚拟变量与多项式一起纳入被筛选的控制集。比较两个结果可以看出，变量选择与高维灵活控制会明显改变对 NTV 报道效应的估计与显著性。

In [ ]:
%%stata
dsregress Votes_SPS_ Watch_probit if year==1999, controls((i.region) `E_controls') cluster(region) selec(cv)